# Libraries import

In [15]:
import chromadb
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# connection t vector DB

In [16]:
client = chromadb.PersistentClient(path="chroma_storage")

collection = client.get_or_create_collection(name="memory_collection")

In [17]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 462.14it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [24]:
def store_memory(text:str):
    embedding=model.encode([text]).tolist()
    collection.add(
        documents=[text],
        embeddings=embedding,
        ids=[f"id{collection.count()+1}"]
    )

In [18]:
def retrive_context(question:str):
    query_emb=model.encode([question]).tolist()
    results=collection.query(
        query_embeddings=query_emb,
        n_results=2
    )
    retrived_docs=results["documents"][0]
    return "\n".join(retrived_docs)
    

In [19]:
retriver=RunnableLambda(retrive_context)

In [20]:
load_dotenv()
llm=ChatGroq(
    model="llama-3.1-8b-instant"
)

In [21]:
prompt=ChatPromptTemplate.from_template(""" 
You are a friendly AI Comapnion.
Use the following memories when answering.
Memories:
{context}
                                        
User message:
{question}                                        
""")

In [22]:
chain=(
    {
        "context":retriver,
        "question":RunnablePassthrough()
    } | prompt | llm
)

In [25]:
def ask_ai(question:str):
    response=chain.invoke(question)
    answer=response.content
    store_memory(f"user said: {question}")
    store_memory(f"Ai replied: {answer}")
    return answer